# Notebook 18 — WINDOW-E3-001: contextual statistics and randomized-history control

**Purpose:** E2 showed that giving a tabular classifier all five serialized records improved accuracy substantially relative to only the newest record. E3 asks **why**: do summary statistics add anything, and does the apparent five-record benefit survive destroying the relationship between the newest record and its historical neighbors?

**Three preregistered experiments:**
1. **stats_unweighted:** Native LightGBM with the exact 210 E2 raw-window features plus six statistics on each of 39 numerical fields (mean, standard deviation, minimum, maximum, oldest-to-target difference, previous-four-mean-to-target difference), **444 total features**.
2. **stats_rare_sqrt:** Same 444 inputs with E2's exact sqrt-balancing class weights.
3. **shuffled_rare_sqrt:** **210 raw features** with each historical 42-field record at lag 1–4 independently permuted across windows **within each split**; target lag0 row, class label, record-level coherence and lag-wise marginals stay fixed. Same rare-weighted LightGBM settings. This negative control tests whether useful alignment between context and target matters; it cannot by itself establish true chronology.

The script first restores the **two actual frozen E2 fitted boosters and predictions** from their SHA256-verified unpublished GitHub Releases; it also restores the original SCV frozen targets and old LightGBM control checkpoints. We **never retrain E2**. The official UNSW-NB15 test set is NEVER opened. We evaluate only the exact same 35,005 original internal-validation target rows as E2. CSV row order is **not verified chronological**.

**Crash-safe archival:** Full CPU training bars, intermediate *local* model snapshots each 100 boosting rounds, and automatic GitHub private draft backup immediately after **each completed candidate**, before the next starts. Genuine per-record predictions and models stay in *draft releases only*. Public repo receives independently recomputed numeric performance, per-class tables and confusion matrices. A new Kaggle cloud session restores completed private candidates rather than training them again. If Kaggle crashes *before completion/upload* of a candidate, its local intermediate booster is not guaranteed to survive.

**Kaggle setup:** Import this notebook, attach `dsshayan/unsw-nb15`, Internet ON, authorize the existing secret `pushRare`, **Accelerator: None (CPU)**. Start **Save Version → Save & Run All**. Final log marker: `WINDOW-E3-001 RESULTS AND PRIVATE BACKUPS COMPLETE`.

**Scientific decision rule:** Compare E3 variants with both frozen E2 models and the 42-feature last-record baselines. Do not claim test-set superiority, true temporal modeling, or a new journal-worthy architecture based on the internal-validation numbers alone.

In [ ]:
# ONE CELL — frozen control restoration, three planned E3 CPU fits, private backups
import os, sys, subprocess, hashlib, json
from pathlib import Path
from kaggle_secrets import UserSecretsClient

REPO=Path("/kaggle/working/rare-attack-ids-window-e3")
BASE=Path("/kaggle/working/research_outputs")
RUN=BASE/"WINDOW-E3-001-S42"
E2=BASE/"WINDOW-E2-001-S42"
TRAIN=Path("/kaggle/input/datasets/dsshayan/unsw-nb15/UNSW_NB15_training-set.csv")
assert TRAIN.is_file(), "Attach dataset dsshayan/unsw-nb15 in Kaggle input"
with TRAIN.open("rb") as handle:
    assert hashlib.file_digest(handle, "sha256").hexdigest()==(
        "bec7dd5ec88dc2a0ccc7a07879d338395ed7421750f675fd0339e07dfe0648fa"
    ),"Unexpected TRAIN CSV content; refusing to change frozen experimental inputs"
token=UserSecretsClient().get_secret("pushRare")
assert token, "Authorize existing Kaggle GitHub secret pushRare in Notebook 18"
if REPO.is_dir():
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
else:
    subprocess.run(["git","clone","--depth","1",
        "https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git",str(REPO)],check=True)

env=os.environ.copy()
env["GITHUB_TOKEN"]=token
env["PYTHONPATH"]=str(REPO/"src")
env["OMP_NUM_THREADS"]="4"
env["OPENBLAS_NUM_THREADS"]="1"
print("E3 pinned GitHub source commit:",
      subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],
                              text=True).strip(),flush=True)

for script,archive in [
    ("restore_scv_release.py","SCV-IDS-REPRO-001_FULL_ARTIFACTS.tar"),
    ("restore_ctrl003_tabular.py","CTRL-TAB-001-S42_FULL_ARTIFACTS.tar"),
]:
    subprocess.run([sys.executable,str(REPO/"scripts"/script),
        "--base",str(BASE),"--archive",str(Path("/kaggle/working")/archive)],
        cwd=str(REPO),env=env,check=True)

# Both E2 fitted models were previously uploaded as private draft GitHub Releases.
# Restore their TRUE 35,005 validation predictions and SHA256-verified checkpoints.
sys.path.insert(0,str(REPO/"scripts"))
from archive_window_e2 import restore_prior_candidate, validate
for model in ("window_unweighted","window_rare_sqrt"):
    restore_prior_candidate(E2,model,token)
    validate(E2,model)
    print("E2 model securely restored WITHOUT retraining:",model,flush=True)

subprocess.run([
    sys.executable,str(REPO/"scripts/run_window_e3.py"),
    "--repo",str(REPO),"--reference",str(BASE/"SCV-IDS-REPRO-001"),
    "--baseline",str(BASE/"CTRL-TAB-001-S42"),
    "--e2",str(E2),"--config",str(REPO/"configs/window_e3_001_seed42.json"),
    "--output",str(RUN),"--auto-archive",
],cwd=str(REPO),env=env,check=True)

print("\nFINAL E3 RESULTS (real internal-validation only):",flush=True)
print((RUN/"comparison_all_models.csv").read_text(),flush=True)
print("Confirmed: all E3 fitted models privately backed up to GitHub.",flush=True)
